# Groq Reasoning Debugging Tutor
######
In this notebook, we will compare two Groq-hosted model sizes and see how reasoning changes the reply on the same debugging task.

## 1. Setup and imports
We need four things for this notebook:

- `os` to read environment variables
- `load_dotenv` to load the API key from a `.env` file
- `OpenAI` to call Groq through its OpenAI-compatible endpoint
- `gradio` to build the chat interface


In [1]:
# %pip -q install gradio openai python-dotenv

import os  # works with environment variables and file paths
import time  # measures latency for each model response
import gradio as gr  # builds the web chat interface
from openai import OpenAI  # provides an OpenAI-compatible API client
from dotenv import load_dotenv  # loads secrets from a local .env file

## 2. Connect to Groq

This cell loads `GROQ_API_KEY` and creates the API client.
The Groq endpoint is OpenAI-compatible, so the same Python client can be reused with a different base URL.

This notebook keeps two model choices:
- `GPT-OSS 20B`
- `GPT-OSS 120B`

The main comparison is model size and reasoning on the same prompt.


In [2]:
api_client = None
api_models = {
    "GPT-OSS 20B": "openai/gpt-oss-20b",
    "GPT-OSS 120B": "openai/gpt-oss-120b",
}

load_dotenv(".env")
api_key = os.getenv("GROQ_API_KEY")
print("GROQ_API_KEY:", "loaded" if api_key else "NOT FOUND")

if api_key:
    api_client = OpenAI(
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )
    print("Groq client ready.")
else:
    print("Add GROQ_API_KEY to your environment or .env file and rerun this cell.")

GROQ_API_KEY: loaded
Groq client ready.


## 3. Blueprint

This section defines the tutoring behavior before any reply is generated.

Four pieces matter here:
- `system_prompt` sets the Socratic tutoring rules
- `reasoning_prompt` adds extra instruction when reasoning is on
- `few_shot_example` shows the style of answer to imitate
- `test_cases` provides repeatable debugging prompts for comparison

The goal is to keep the prompt fixed while the model and reasoning setting change.


In [3]:
app_title = "AI Debugging Tutor — Groq Reasoning"
app_desc = (
    "A notebook-only Groq debugging tutor for comparing two model sizes, "
    "optional reasoning, and few-shot prompting."
)

system_prompt = '''
You are a Socratic debugging tutor.

Rules:
- Do not give the final answer or full corrected code.
- Ask one small guiding question at a time.
- Give only the next hint or check.
- Keep answers short and clear.
- If the user asks for the answer, do not give it.
'''

reasoning_prompt = '''
Think through the bug carefully before giving the next hint.
Keep the final reply clear.
'''

few_shot_example = '''
Example 1:
User:
age = 25
print("I am " + age)

Assistant:
What is the type of `age` right now? Can `+` join a string and an integer directly?

Example 2:
User:
Please just give me the answer.

Assistant:
Before we jump to the answer, what part feels most confusing right now: the error message, the variable type, or the loop logic?
'''

test_cases = {
    "1. CSV Loading [Concept]": {
        "input": "How do I read a CSV file with the datascience package?"
    },
    "2. TypeError [Easy]": {
        "input": "age = 25\nprint(\"I am \" + age)\n\nTypeError: can only concatenate str to str"
    },
    "3. Average Score [Logic]": {
        "input": (
            "def average_score(scores):\n"
            "    total = 0\n"
            "    for score in scores:\n"
            "        total = score\n"
            "    return total / len(scores)\n\n"
            "print(average_score([80, 90, 70, 100]))"
        )
    },
    "4. Prefix Sum [Index]": {
        "input": (
            "nums = [3, 1, 4, 2]\nprefix = []\ncurrent = 0\n\n"
            "for i in range(len(nums) + 1):\n"
            "    current += nums[i]\n"
            "    prefix.append(current)\n\n"
            "print(prefix)"
        )
    },
}


## 4. State

State means data that stays available while the app is running.

This notebook stores `run_log`, which records the model choice, reasoning setting, token counts, and latency from recent runs.
That log makes it easier to compare four runs of the same test case side by side.


In [4]:
run_log = []

## 5. Helper functions

These functions handle the main logic of the tutor.

The important comparison happens in two places:
- `model_name` changes the model size
- `use_reasoning` changes whether extra reasoning effort is requested

The `Context` panel shows the exact prompt sent to Groq.
The experiment log records the settings and measurements from each run.
That makes it easier to compare small versus large models and reasoning off versus on.


In [5]:
def clean_message(message):
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: " )[0]

    return {"role": role, "content": text}


def build_prompt(system_prompt, reasoning_prompt, few_shot_example, use_reasoning, use_few_shot):
    active_prompt = system_prompt.strip()
    if use_reasoning:
        active_prompt += "\n\n" + reasoning_prompt.strip()
    if use_few_shot:
        active_prompt += "\n\n" + few_shot_example.strip()
    return active_prompt


def build_query(user_input, test_case, chat_history):
    if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip():
            return user_input.strip()
        return test_cases[test_case]["input"]
    return user_input.strip()


def count_text_tokens(text):
    return max(1, len(str(text).split()))


def build_messages(active_prompt, chat_history):
    messages = [{"role": "system", "content": active_prompt}]
    for message in chat_history:
        messages.append(clean_message(message))
    return messages


def build_context_text(messages):
    parts = []
    for message in messages:
        parts.append(f"[{message['role'].upper()}]")
        parts.append(str(message["content"]))
        parts.append("-" * 40)
    return "\n".join(parts)


def build_extra_body(use_reasoning, reasoning_level):
    if not use_reasoning:
        return {"reasoning_effort": "none", "include_reasoning": False}

    selected_level = str(reasoning_level).lower()
    if selected_level not in {"low", "medium", "high"}:
        selected_level = "low"

    return {
        "reasoning_effort": selected_level,
        "include_reasoning": False,
    }


def generate_response(messages, model_name, temperature, max_tokens, use_reasoning, reasoning_level):
    if api_client is None:
        final_text = "Groq client is not ready. Load GROQ_API_KEY and rerun the connection cell."
        return final_text, None, count_text_tokens(final_text)

    try:
        response = api_client.chat.completions.create(
            model=api_models[model_name],
            messages=messages,
            temperature=float(temperature),
            max_completion_tokens=int(max_tokens),
            top_p=0.95,
            extra_body=build_extra_body(use_reasoning, reasoning_level),
        )
        final_text = response.choices[0].message.content or ""
        if final_text == "":
            final_text = "[No text returned]"

        usage = getattr(response, "usage", None)
        prompt_tokens = getattr(usage, "prompt_tokens", None) if usage is not None else None
        output_tokens = getattr(usage, "completion_tokens", None) if usage is not None else None
        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)
        return final_text, prompt_tokens, output_tokens
    except Exception as error:
        final_text = "Generation failed: " + str(error)
        return final_text, None, count_text_tokens(final_text)


def build_log_rows(run_log):
    rows = []
    for record in run_log[-10:]:
        rows.append([
            record["run"],
            record["test case"],
            record["model"],
            record["reasoning"],
            record["few_shot"],
            record["input_tokens"],
            record["output_tokens"],
            record["latency"],
        ])
    return rows


def run_chatbot(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_history,
):
    global run_log

    if chat_history is None:
        chat_history = []

    active_prompt = build_prompt(
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        use_few_shot,
    )
    messages = build_messages(active_prompt, chat_history)
    user_query = build_query(user_input, test_case, chat_history)

    if user_query == "":
        return chat_history, "", build_log_rows(run_log), "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_query})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["input"]
        else:
            display_user = "[" + test_case + "]"

    chat_history = list(chat_history)
    chat_history.append({"role": "user", "content": display_user})
    chat_history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, prompt_tokens, output_tokens = generate_response(
        messages,
        model_name,
        temperature,
        max_tokens,
        use_reasoning,
        reasoning_level,
    )
    latency = round(time.perf_counter() - start_time, 2)

    input_tokens = prompt_tokens
    if input_tokens is None:
        input_tokens = count_text_tokens(context_text)

    reasoning_label = str(reasoning_level).lower() if use_reasoning else "off"
    badge = (
        f"\n\n`Model: {model_name}"
        f" | Time: {latency}s"
        f" | Input tokens: {input_tokens}"
        f" | Output tokens: {output_tokens}"
        f" | Reasoning: {reasoning_label}`"
    )
    chat_history[-1]["content"] = final_text + badge

    run_log.append({
        "run": len(run_log) + 1,
        "test case": test_case,
        "model": model_name,
        "reasoning": reasoning_label,
        "few_shot": "On" if use_few_shot else "Off",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
    })

    return (
        chat_history,
        context_text,
        build_log_rows(run_log),
        "",
        "**Status:** Done",
    )


def clear_chat():
    global run_log
    run_log = []
    return [], "", [], "", "**Status:** Ready"


def load_test_case(test_case):
    if test_case == "Free Typing":
        return "", "**Status:** Ready"
    return test_cases[test_case]["input"], "**Status:** Loaded " + test_case


def run_chatbot_ui(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_state,
):
    chat_history, context_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        temperature,
        max_tokens,
        test_case,
        model_name,
        chat_state,
    )
    return (
        chat_history,
        context_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


def clear_chat_ui():
    chat_history, context_text, log_rows, next_input, status_message = clear_chat()
    return (
        chat_history,
        context_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


## 6. Build the interface

This section builds the Gradio app.

The chat appears on the left.
The controls appear on the right.
The bottom panels show the exact context and a small experiment log.

This notebook is meant to be used as a comparison tool.
The same test case should be run several times while changing only one control at a time.


In [13]:
ui_test_cases = ["Free Typing"] + list(test_cases.keys())
ui_models = list(api_models.keys())

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")

    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(label="Chat", height=557)
            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your debugging question here...",
                    lines=4,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_test_cases,
                value="Free Typing",
                label="Test Case",
            )
            model_dropdown = gr.Dropdown(
                choices=ui_models,
                value=ui_models[0] if ui_models else None,
                label="Groq Model",
            )
            use_reasoning = gr.Checkbox(label="Use Reasoning", value=False)
            reasoning_level = gr.Dropdown(
                choices=["low", "medium", "high"],
                value="low",
                label="Reasoning Level",
            )
            use_few_shot = gr.Checkbox(label="Use Few-Shot", value=False)

            with gr.Accordion("Model Settings", open=True):
                temperature_slider = gr.Slider(0.0, 1.0, step=0.1, value=0.2, label="Temperature")
                max_tokens_slider = gr.Slider(64, 2048, step=64, value=512, label="Max Tokens")

            with gr.Accordion("Edit Prompts", open=False):
                system_prompt_box = gr.Textbox(label="System Prompt", value=system_prompt, lines=8)
                reasoning_prompt_box = gr.Textbox(
                    label="Reasoning Prompt",
                    value=reasoning_prompt,
                    lines=5,
                )
                few_shot_box = gr.Textbox(label="Few-Shot Example", value=few_shot_example, lines=8)

    with gr.Row():
        context_box = gr.Textbox(label="Context", value="", lines=16, interactive=False)

    log_table = gr.Dataframe(
        headers=["Run", "Test Case", "Model", "Reasoning", "Few-Shot", "Input", "Output", "Latency"],
        datatype=["number", "str", "str", "str", "str", "number", "number", "number"],
        row_count=10,
        column_count=(8, "fixed"),
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    inputs = [
        user_input_box,
        system_prompt_box,
        reasoning_prompt_box,
        few_shot_box,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        model_dropdown,
        chat_state,
    ]

    outputs = [
        chat_window,
        context_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    test_case_dropdown.change(load_test_case, test_case_dropdown, [user_input_box, status_box])
    ask_button.click(run_chatbot_ui, inputs, outputs)
    user_input_box.submit(run_chatbot_ui, inputs, outputs)
    clear_button.click(clear_chat_ui, None, outputs)


## 7. Run the app

This cell launches the Groq web app.

A good workflow is:
1. Pick one test case.
2. Run it with `GPT-OSS 20B` and reasoning off.
3. Clear the chat and run the same test case with `GPT-OSS 20B` and reasoning on.
4. Clear the chat and repeat the same two runs with `GPT-OSS 120B`.
5. Compare the four results in the log.

Things to compare:
- whether the larger model gives a more stable hint
- whether reasoning produces a more step-by-step reply
- whether latency and token use increase when reasoning is on
- whether few-shot prompting changes the tutoring style across both models


In [14]:
web_app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7869
* Running on public URL: https://6fb933cf6fbf852ef7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Summary

This notebook connected to Groq, kept two model sizes in the interface, added optional reasoning controls, and launched a debugging tutor for side-by-side comparison.
The main comparison in this version is model size and reasoning on the same task.
That makes it easier to see how answer depth, latency, and token use change when the backend setting changes.
